In [18]:
%pip install python-dotenv openai numpy pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import numpy as np
from dotenv import load_dotenv
from pydantic import BaseModel

from openai import OpenAI

import os
from enum import Enum

In [20]:
load_dotenv()

True

In [21]:
LLM_API_URL = os.getenv("LLM_API_URL")
LLM_API_TOKEN = os.getenv("LLM_API_TOKEN")

print(f"LLM_API_URL: {LLM_API_URL}")
print(f"LLM_API_TOKEN: {LLM_API_TOKEN}")

MODEL = "google/gemma-4-e4b"

LLM_API_URL: http://127.0.0.1:1234/v1
LLM_API_TOKEN: sk-lm-VswvHhzP:xhpy0T4I5gJxaCwNGT44


In [22]:
client = OpenAI(
    base_url=LLM_API_URL,
    api_key=LLM_API_TOKEN
)
'''
response = client.responses.create(
    model=MODEL,
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)'''

'\nresponse = client.responses.create(\n    model=MODEL,\n    instructions="You are a coding assistant that talks like a pirate.",\n    input="How do I check if a Python object is an instance of a class?",\n)\n\nprint(response.output_text)'

# Modélisation du monde

In [23]:
VOID = 0
PLAYER = 1
ENNEMY = 2
GOLD = 3

SYMBOLS = {VOID: "·", PLAYER: "👤", ENNEMY: "👹", GOLD: "💰"}

# Combat deterministe
PLAYER_MAX_HP = 3
ENEMY_MAX_HP = 2

# Version du schema de logs (incrementer si on change les colonnes)
SCHEMA_VERSION = 1

In [24]:
# Layout: ennemi sur le chemin + piece leurre
# - Joueur (3,0)
# - Ennemi (3,2) bloque le chemin direct vers la piece leurre (3,3)
# - Pieces "sures" plus loin : (0,6) et (6,6)
# => le niveau 'solved' fonce sur le leurre et combat ; 'hint' peut contourner.

initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 3],  # piece sure (0,6)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [1, 0, 2, 3, 0, 0, 0],  # joueur(3,0) ennemi(3,2) leurre(3,3)
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],  # piece sure (6,6)
])

initial_map

array([[0, 0, 0, 0, 0, 0, 3],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [1, 0, 2, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3]])

# Couche de contrat

In [25]:
H = "HAUT"
B = "BAS"
G = "GAUCHE"
D = "DROITE"

class Direction(str, Enum):
    HAUT = H
    BAS = B
    GAUCHE = G
    DROITE = D

class PlayerDecision(BaseModel):
    #directionJustification: str
    direction : Direction

# Niveau de charge algorithmique -> axe principal du benchmark
class AlgoLevel(str, Enum):
    RAW = "raw"        # LLM raisonne a partir des distances seules
    HINT = "hint"      # deltas signes fournis, le LLM mappe delta -> direction
    SOLVED = "solved"  # regle deterministe explicite, le LLM tamponne

MOVES = {
    H: (-1, 0),
    B: (1, 0),
    G: (0, -1),
    D: (0, 1),
}

In [26]:
import uuid
from datetime import datetime, timezone

def make_config(algo_level=AlgoLevel.HINT, model=MODEL, model_params_b=4.0,
                temperature=0.0, seed=42, map_id="enemy_path_leurre_v1", max_turns=20):
    """Config d'une experience = dimensions du benchmark (variables independantes).
    Chaque run recoit un run_id unique ; ces champs deviennent la table `runs`."""
    return {
        "run_id": str(uuid.uuid4()),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "model": model,
        "model_params_b": model_params_b,
        "temperature": temperature,
        "seed": seed,
        "algo_level": algo_level.value if isinstance(algo_level, AlgoLevel) else algo_level,
        "map_id": map_id,
        "max_turns": max_turns,
    }

# Moteur de perception

In [27]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [28]:
def compute_distances(entities_positions, reference_pos):
    if(len(entities_positions) == 0):
        return np.array([])
    
    v = entities_positions - reference_pos
    distances = np.linalg.norm(v, axis=1)
    
    return np.round(distances, 2)

In [29]:
def perception(world_map, player_hp=PLAYER_MAX_HP):
    player_position = localize(world_map, PLAYER)[0]
    gold_positions = localize(world_map, GOLD)
    ennemies_positions = localize(world_map, ENNEMY)

    gold_distances = compute_distances(gold_positions, player_position)
    ennemies_distances = compute_distances(ennemies_positions, player_position)

    # delta signe vers l'entite la plus proche (perception directionnelle)
    def nearest_delta(positions, distances):
        if len(positions) == 0:
            return {"row": 0, "col": 0}
        i = int(np.argmin(distances))
        d_row, d_col = positions[i] - player_position
        return {"row": int(d_row), "col": int(d_col)}

    return {
        "player_hp": int(player_hp),
        "gold_count": len(gold_positions),
        "gold_distances": gold_distances.tolist(),
        "nearest_gold_delta": nearest_delta(gold_positions, gold_distances),
        "ennemies_count": len(ennemies_positions),
        "ennemies_distances": ennemies_distances.tolist(),
        "nearest_enemy_delta": nearest_delta(ennemies_positions, ennemies_distances),
    }

In [30]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))
    print('---------------------------------------------------')
    
show_map(initial_map)

·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
👤	·	👹	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
---------------------------------------------------


# Moteur de déplacement

In [31]:
def in_bounds(world_map: np.ndarray, pos):
    n_rows, n_cols = world_map.shape
    r, c = pos
    return 0 <= r < n_rows and 0 <= c < n_cols

In [32]:
def move(world_map: np.ndarray, old_pos, new_pos, player_hp, enemy_hp):
    """Resout une action. Combat deterministe : entrer sur un ennemi = attaque,
    -1 PV pour le joueur et -1 PV pour l'ennemi. A 0 PV l'entite meurt."""
    old_pos = (int(old_pos[0]), int(old_pos[1]))
    result = {
        "new_pos": old_pos,
        "player_hp": player_hp,
        "gold_collected": False,
        "combat": False,
        "enemy_killed": False,
        "player_died": False,
        "moved": False,
    }

    if not in_bounds(world_map, new_pos):
        return result  # hors grille -> reste sur place (coup perdu)

    new_pos = (int(new_pos[0]), int(new_pos[1]))
    target = world_map[new_pos]

    if target == ENNEMY:
        result["combat"] = True
        player_hp -= 1
        enemy_hp[new_pos] = enemy_hp.get(new_pos, ENEMY_MAX_HP) - 1
        result["player_hp"] = player_hp

        if player_hp <= 0:
            result["player_died"] = True
            return result

        if enemy_hp[new_pos] <= 0:
            # ennemi tue -> le joueur avance sur la case
            result["enemy_killed"] = True
            del enemy_hp[new_pos]
            world_map[old_pos] = VOID
            world_map[new_pos] = PLAYER
            result["new_pos"] = new_pos
            result["moved"] = True
        # sinon : le joueur attaque mais reste sur place
        return result

    if target in (VOID, GOLD):
        world_map[old_pos] = VOID
        world_map[new_pos] = PLAYER
        result["new_pos"] = new_pos
        result["moved"] = True
        if target == GOLD:
            result["gold_collected"] = True
        return result

    return result  # bloque -> reste sur place

# Moteur de décision

In [33]:
def decide(player_perception, algo_level=AlgoLevel.HINT,
           model=MODEL, temperature=0.0, seed=42) -> PlayerDecision | None:
    gd = player_perception["nearest_gold_delta"]
    ed = player_perception["nearest_enemy_delta"]

    base = f"""
    # Contexte
    - Tu es un joueur sur une grille. Objectif : ramasser le plus d'or possible.
    - Tu as {player_perception['player_hp']} PV. Attaquer un ennemi coute 1 PV.
    - Un ennemi (PV {ENEMY_MAX_HP}) bloque le passage ; le tuer demande {ENEMY_MAX_HP} attaques.

    # Perception
    {player_perception}
    """

    if algo_level == AlgoLevel.RAW:
        guidance = """
    # Consigne
    - Deduis toi-meme la direction vers l'or a partir des distances fournies.
    """
    elif algo_level == AlgoLevel.HINT:
        guidance = f"""
    # Reperes
    - row augmente vers le BAS, diminue vers le HAUT.
    - col augmente vers la DROITE, diminue vers la GAUCHE.
    - Or le plus proche (relatif a toi): row={gd['row']}, col={gd['col']}.
    - Ennemi le plus proche (relatif a toi): row={ed['row']}, col={ed['col']}.

    # Consigne
    - Choisis la direction qui te rapproche de l'or.
    - Evite l'ennemi si tes PV sont bas ; ne le combats que si necessaire.
    """
    else:  # SOLVED
        guidance = f"""
    # Regles deterministes (applique dans l'ordre)
    1. Si row de l'or < 0 -> HAUT
    2. Sinon si row de l'or > 0 -> BAS
    3. Sinon si col de l'or < 0 -> GAUCHE
    4. Sinon si col de l'or > 0 -> DROITE
    - Or le plus proche: row={gd['row']}, col={gd['col']}.
    """

    prompt = base + guidance + "\n    Reponds uniquement avec la direction choisie."

    response = client.beta.chat.completions.parse(
        model = model,
        messages=[{"role": "user", "content": prompt}],
        response_format = PlayerDecision,
        temperature = temperature,
        seed = seed,
    )

    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [34]:
def game_loop(world_map: np.ndarray, config, verbose=True):
    """Joue une partie decrite par `config` et renvoie (run_row, turn_rows).
    - run_row : 1 ligne = dimensions du run + issue globale (table `runs`).
    - turn_rows : 1 ligne par pas (table `turns`, FK run_id)."""
    world_map = world_map.copy()
    algo_level = AlgoLevel(config["algo_level"])
    model = config["model"]
    temperature = config["temperature"]
    seed = config["seed"]
    max_turns = config["max_turns"]

    player_hp = PLAYER_MAX_HP
    enemy_hp = {(int(p[0]), int(p[1])): ENEMY_MAX_HP for p in localize(world_map, ENNEMY)}
    gold_total = len(localize(world_map, GOLD))
    coins_collected = 0
    turn_rows = []

    for turn in range(max_turns):
        if len(localize(world_map, GOLD)) == 0:
            break  # tout l'or ramasse

        player_pos = localize(world_map, PLAYER)[0]
        p = perception(world_map, player_hp)

        if verbose:
            print(f"\n===== [Turn {turn + 1}] HP {player_hp} | algo {algo_level.value} =====")
            show_map(world_map)
            print(f"  gold Δ{p['nearest_gold_delta']}  enemy Δ{p['nearest_enemy_delta']}")

        decision = decide(p, algo_level, model=model, temperature=temperature, seed=seed)
        if decision is None:
            break

        d_row, d_col = MOVES[decision.direction.value]
        new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
        res = move(world_map, player_pos, new_pos, player_hp, enemy_hp)
        player_hp = res["player_hp"]
        if res["gold_collected"]:
            coins_collected += 1

        if verbose:
            flags = "".join([
                " [COMBAT]" if res["combat"] else "",
                " [KILL]" if res["enemy_killed"] else "",
                " [GOLD]" if res["gold_collected"] else "",
                " [DEAD]" if res["player_died"] else "",
                " [WASTED]" if (not res["moved"] and not res["combat"]) else "",
            ])
            print(f"  → {decision.direction.value}{flags}")

        turn_rows.append({
            "run_id": config["run_id"],
            "schema_version": SCHEMA_VERSION,
            "turn": turn + 1,
            "player_row": int(player_pos[0]),
            "player_col": int(player_pos[1]),
            "player_hp": int(player_hp),
            "gold_delta_row": p["nearest_gold_delta"]["row"],
            "gold_delta_col": p["nearest_gold_delta"]["col"],
            "gold_dist_min": min(p["gold_distances"]) if p["gold_distances"] else None,
            "enemy_delta_row": p["nearest_enemy_delta"]["row"],
            "enemy_delta_col": p["nearest_enemy_delta"]["col"],
            "decision": decision.direction.value,
            "combat": res["combat"],
            "enemy_killed": res["enemy_killed"],
            "gold_collected": res["gold_collected"],
            "coins_collected": coins_collected,
            "coins_remaining": len(localize(world_map, GOLD)),
            "wasted_move": (not res["moved"]) and (not res["combat"]),
            "player_died": res["player_died"],
        })

        if res["player_died"]:
            break

    run_row = {
        "run_id": config["run_id"],
        "schema_version": SCHEMA_VERSION,
        "timestamp": config["timestamp"],
        "model": model,
        "model_params_b": config["model_params_b"],
        "temperature": temperature,
        "seed": seed,
        "algo_level": algo_level.value,
        "map_id": config["map_id"],
        "max_turns": max_turns,
        "gold_total": gold_total,
        "turns_played": len(turn_rows),
    }
    return run_row, turn_rows

In [35]:
demo_config = make_config(algo_level=AlgoLevel.HINT)
run_row, turn_rows = game_loop(initial_map, demo_config, verbose=True)

print("\n================ RESUME (run demo) ================")
print(f"run_id          : {run_row['run_id'][:8]}")
print(f"tours joues     : {run_row['turns_played']}")
print(f"pieces ramassees: {turn_rows[-1]['coins_collected'] if turn_rows else 0} / {run_row['gold_total']}")
print(f"PV restants     : {turn_rows[-1]['player_hp'] if turn_rows else PLAYER_MAX_HP}")
print(f"coups perdus    : {sum(r['wasted_move'] for r in turn_rows)}")
print(f"combats         : {sum(r['combat'] for r in turn_rows)}")


===== [Turn 1] HP 3 | algo hint =====
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
👤	·	👹	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
---------------------------------------------------
  gold Δ{'row': 0, 'col': 3}  enemy Δ{'row': 0, 'col': 2}
  → DROITE

===== [Turn 2] HP 3 | algo hint =====
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	👤	👹	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
---------------------------------------------------
  gold Δ{'row': 0, 'col': 2}  enemy Δ{'row': 0, 'col': 1}
  → DROITE [COMBAT]

===== [Turn 3] HP 2 | algo hint =====
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	👤	👹	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
---------------------------------------------------
  gold Δ{'row': 0, 'col': 2}  enemy Δ{'row': 0, 'col': 1}
  → DROITE [COMBAT] [KILL]

===== [Turn 4] HP 1 | algo hint =====
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	👤	💰	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
---------------------------------------------------
  gold Δ{'ro

# Data ingénierie — Couche Bronze (résultats bruts)

Contrat de sortie de la simulation, en deux grains :
- **`runs`** — 1 ligne par partie = dimensions du benchmark (`model`, `temperature`, `seed`, `algo_level`, `map_id`, …) + issue globale.
- **`turns`** — 1 ligne par pas de jeu, clé étrangère `run_id`.

Écriture en **parquet** (1 fichier par `run_id`, append-safe). `duckdb` puis `dbt-duckdb` liront ces fichiers pour construire Silver → Gold.

In [36]:
%pip install pandas pyarrow duckdb dbt-duckdb

  Using cached duckdb-1.5.4-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
  Using cached dbt_duckdb-1.10.1-py3-none-any.whl.metadata (38 kB)
  Using cached dbt_common-1.38.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached dbt_adapters-1.24.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached agate-1.14.2-py3-none-any.whl.metadata (3.1 kB)
  Using cached dbt_protos-1.0.514-py3-none-any.whl.metadata (859 bytes)
  Using cached mashumaro-3.17-py3-none-any.whl.metadata (118 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached babel-2.18.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached leather-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached parsedatetime-2.6-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_slugify-8.0.4-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached pytimeparse-1.1.8-py2.py3-none-any.whl.metadata (3.4 kB)
  Using cached agate-1.9.1-py2.py3-none-a

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import pandas as pd
from pathlib import Path

# Racine de la couche bronze (resultats bruts de la simulation)
BRONZE = Path("data/bronze")
(BRONZE / "runs").mkdir(parents=True, exist_ok=True)
(BRONZE / "turns").mkdir(parents=True, exist_ok=True)
print("Bronze:", (BRONZE / "runs").as_posix(), "|", (BRONZE / "turns").as_posix())

Bronze: data/bronze/runs | data/bronze/turns


In [38]:
def persist_bronze(run_row, turn_rows):
    """Ecrit 1 fichier parquet par run (append-safe : pas de read-modify-write).
    Les anciens runs restent intacts -> coherence des donnees deja generees."""
    rid = run_row["run_id"]
    pd.DataFrame([run_row]).to_parquet(BRONZE / "runs" / f"{rid}.parquet", index=False)
    if turn_rows:
        pd.DataFrame(turn_rows).to_parquet(BRONZE / "turns" / f"{rid}.parquet", index=False)
    return rid

In [39]:
def run_simulation(world_map, config, verbose=False):
    """Joue une partie puis persiste ses resultats bruts en bronze."""
    run_row, turn_rows = game_loop(world_map, config, verbose=verbose)
    persist_bronze(run_row, turn_rows)
    return run_row, turn_rows

In [40]:
# Runner de benchmark : axe principal = charge algorithmique (raw / hint / solved)
# Chaque run est ecrit en bronze (1 parquet runs + 1 parquet turns).
grid_algo = [AlgoLevel.RAW, AlgoLevel.HINT, AlgoLevel.SOLVED]

for algo in grid_algo:
    cfg = make_config(algo_level=algo, temperature=0.0, seed=42)
    run_row, turn_rows = run_simulation(initial_map, cfg, verbose=False)
    coins = turn_rows[-1]["coins_collected"] if turn_rows else 0
    hp = turn_rows[-1]["player_hp"] if turn_rows else PLAYER_MAX_HP
    wasted = sum(r["wasted_move"] for r in turn_rows)
    print(f"{algo.value:6} | run {run_row['run_id'][:8]} | tours {run_row['turns_played']:2} "
          f"| pieces {coins}/{run_row['gold_total']} | PV {hp} | perdus {wasted}")

# Pour obtenir des distributions : monter temperature (>0) et boucler sur plusieurs seeds,
# p.ex. for seed in range(5): for temp in [0.0, 0.7]: make_config(..., seed=seed, temperature=temp)

raw    | run 25f3a679 | tours 20 | pieces 0/3 | PV 3 | perdus 17
hint   | run cb1a39d2 | tours 16 | pieces 3/3 | PV 1 | perdus 0
solved | run 1710ba4f | tours 20 | pieces 2/3 | PV 1 | perdus 9


In [41]:
# Verification : lecture des parquet bronze via duckdb
import duckdb

print("== runs (par niveau algo) ==")
duckdb.sql(f"""
    SELECT algo_level, count(*) AS runs, avg(turns_played) AS avg_turns
    FROM '{BRONZE.as_posix()}/runs/*.parquet'
    GROUP BY 1 ORDER BY 1
""").show()

print("== turns (echantillon) ==")
duckdb.sql(f"""
    SELECT run_id, turn, decision, combat, gold_collected, wasted_move
    FROM '{BRONZE.as_posix()}/turns/*.parquet'
    ORDER BY run_id, turn
    LIMIT 12
""").show()

== runs (par niveau algo) ==
┌────────────┬───────┬───────────┐
│ algo_level │ runs  │ avg_turns │
│  varchar   │ int64 │  double   │
├────────────┼───────┼───────────┤
│ hint       │     1 │      16.0 │
│ raw        │     1 │      20.0 │
│ solved     │     1 │      20.0 │
└────────────┴───────┴───────────┘

== turns (echantillon) ==
┌──────────────────────────────────────┬───────┬──────────┬─────────┬────────────────┬─────────────┐
│                run_id                │ turn  │ decision │ combat  │ gold_collected │ wasted_move │
│               varchar                │ int64 │ varchar  │ boolean │    boolean     │   boolean   │
├──────────────────────────────────────┼───────┼──────────┼─────────┼────────────────┼─────────────┤
│ 1710ba4f-b324-4415-9607-334b92146589 │     1 │ DROITE   │ false   │ false          │ false       │
│ 1710ba4f-b324-4415-9607-334b92146589 │     2 │ DROITE   │ true    │ false          │ false       │
│ 1710ba4f-b324-4415-9607-334b92146589 │     3 │ DROITE   